# 🎬 Tạo VIDEO NHẠC bằng AI — Khúc ca Trường Kinh tế

Sinh **hình ảnh bằng AI** (Stable Diffusion) theo chủ đề bài hát, rồi ghép thành **MP4** có hiệu ứng chuyển động + **lời chạy trên màn hình (tiếng Việt CÓ DẤU)**, đồng bộ với bản nhạc.

Chạy **miễn phí trên Google Colab** — bật GPU: *Runtime → Change runtime type → T4 GPU*.

**Đầu vào:** 1 file audio bài hát (lấy từ notebook trước):
- `song_*.wav` = bản **có giọng hát** (AI hát)
- `backing_*.wav` = bản **KHÔNG lời** (chỉ nhạc nền)


## 0. Kiểm tra GPU


In [ ]:
!nvidia-smi -L || print('CHUA BAT GPU: Runtime -> Change runtime type -> T4 GPU')


## 1. Cài đặt (gồm FONT tiếng Việt có dấu)
Cài ffmpeg + font **Be Vietnam Pro** (thiết kế riêng cho tiếng Việt), dự phòng **DejaVu/Noto** nếu tải lỗi.


In [ ]:
!pip -q install diffusers transformers accelerate safetensors 2>/dev/null
!apt -qq install -y ffmpeg fonts-dejavu-core fonts-noto-core >/dev/null 2>&1 || true

# Tai font Be Vietnam Pro (ho tro day du dau tieng Viet)
import os, subprocess
os.makedirs('/usr/share/fonts/truetype/bevn', exist_ok=True)
BASE='https://github.com/google/fonts/raw/main/ofl/bevietnampro'
ok=True
for f in ['BeVietnamPro-Regular.ttf','BeVietnamPro-Bold.ttf']:
    r=subprocess.run(['wget','-q','-O',f'/usr/share/fonts/truetype/bevn/{f}',f'{BASE}/{f}'])
    ok = ok and (r.returncode==0)
!fc-cache -f >/dev/null 2>&1

# Chon font se dung cho phu de
import glob
if ok and glob.glob('/usr/share/fonts/truetype/bevn/*.ttf'):
    FONT_NAME='Be Vietnam Pro'; FONTS_DIR='/usr/share/fonts/truetype/bevn'
else:
    FONT_NAME='DejaVu Sans'; FONTS_DIR='/usr/share/fonts/truetype/dejavu'
print('Font phu de:', FONT_NAME, '| dir:', FONTS_DIR)
import torch; print('CUDA:', torch.cuda.is_available())


## 2. Tải file audio bài hát
Chọn 1 file `.wav` (song_* hoặc backing_*).


In [ ]:
from google.colab import files
up = files.upload()
AUDIO = list(up.keys())[0]
import subprocess
DUR = float(subprocess.check_output(
    ['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',AUDIO]).strip())
print(f'Audio: {AUDIO}  ({DUR:.1f}s)')


## 3. Sinh hình ảnh bằng AI
Chủ đề: trường đại học, sinh viên, quê hương, tự hào — hợp tinh thần **nhạc đỏ**. Sửa danh sách `PROMPTS` tuỳ ý (số ảnh = số prompt).

> Dùng **SDXL-Turbo** (nhanh, vài giây/ảnh trên T4).


In [ ]:
from diffusers import AutoPipelineForText2Image
import torch
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')

STYLE = ', cinematic, warm lighting, high detail, vietnam, proud, hopeful'
PROMPTS = [
  'Vietnamese university campus at sunrise, students walking happily',
  'graduation ceremony, students in caps and gowns celebrating',
  'young Vietnamese students studying together in a bright library',
  'modern Can Tho University building under blue sky',
  'Mekong delta countryside, green rice fields, peaceful river, sunrise',
  'Vietnamese national flag waving in golden light, patriotic',
  'modern Vietnamese city skyline, development and prosperity',
  'many hands joining together, teamwork and unity, warm tone',
]

import os; os.makedirs('frames', exist_ok=True)
paths=[]
for i,p in enumerate(PROMPTS):
    img = pipe(prompt=p+STYLE, num_inference_steps=4, guidance_scale=0.0).images[0]
    img = img.resize((1280,720))
    fp=f'frames/img_{i:03d}.png'; img.save(fp); paths.append(fp)
    print('OK', fp)
print('Tong cong', len(paths), 'anh')


## 4. Tạo file lời chạy (.srt) — UTF-8 có dấu
Chia đều các câu lời theo thời lượng bài. (Video **không lời**: bỏ qua, đặt `BURN_LYRICS=False` ở bước 5.)


In [ ]:
LYRICS_LINES = [
  'Bước trên con đường lòng hân hoan,',
  'ngàn ước mơ xanh dưới mái trường.',
  'Những gian truân nhọc nhằn hôm qua để lại,',
  'cùng đắp xây tương lai bừng sáng.',
  'Non sông đang đổi thay từng ngày,',
  'có chúng tôi chung bàn tay,',
  'cùng mang yên vui đến cho mọi người,',
  'ấm áp trên môi nụ cười.',
  'Cùng dựng xây đất nước phồn vinh muôn đời,',
  'những doanh nghiệp vươn ra thế giới.',
  'Cùng điểm tô quê hương đẹp tươi muôn màu,',
  'và làm nên tổ quốc mạnh giàu.',
  'Trường Kinh tế giữ sứ mệnh ươm nhân tài,',
  'tri thức luyện rèn cho tương lai,',
  'vì cộng đồng sẽ chia giá trị,',
  'và chung tay vun đắp cuộc đời.',
  'Hát lên bạn ơi, khúc ca Trường Kinh tế,',
  'chữ tín ta dựng xây bằng Chất lượng Thân thiện.',
  'Ôi tự hào Trường Kinh tế Đại học Cần Thơ.',
]
def ts(t):
    h=int(t//3600); m=int((t%3600)//60); s=t%60
    return f'{h:02d}:{m:02d}:{s:06.3f}'.replace('.',',')
per = DUR/len(LYRICS_LINES)
with open('lyrics.srt','w',encoding='utf-8') as f:   # UTF-8 giu dau tieng Viet
    for i,line in enumerate(LYRICS_LINES):
        f.write(f'{i+1}\n{ts(i*per)} --> {ts((i+1)*per)}\n{line}\n\n')
print('Da tao lyrics.srt (UTF-8)')


## 5. Ghép thành VIDEO (hiệu ứng zoom + lời có dấu + nhạc)
Phụ đề dùng font `FONT_NAME` đã chọn ở bước 1 → **không mất dấu tiếng Việt**. Đặt `BURN_LYRICS=False` nếu không muốn hiện lời.


In [ ]:
import subprocess, os
BURN_LYRICS = True
N=len(paths); per=DUR/N; fps=30; d=int(round(per*fps))
os.makedirs('clips',exist_ok=True)
for i,fp in enumerate(paths):
    out=f'clips/c_{i:03d}.mp4'
    vf=(f"scale=1600:900:force_original_aspect_ratio=increase,crop=1600:900,"
        f"zoompan=z='min(zoom+0.0012,1.15)':d={d}:s=1280x720:fps={fps},format=yuv420p")
    subprocess.run(['ffmpeg','-y','-loop','1','-i',fp,'-t',f'{per}','-vf',vf,'-r',str(fps),out],
                   check=True, stderr=subprocess.DEVNULL)
with open('list.txt','w') as f:
    for i in range(N): f.write(f"file 'clips/c_{i:03d}.mp4'\n")
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','list.txt','-c','copy','video_mute.mp4'],
               check=True, stderr=subprocess.DEVNULL)
subprocess.run(['ffmpeg','-y','-i','video_mute.mp4','-i',AUDIO,'-c:v','copy','-c:a','aac',
                '-shortest','video_audio.mp4'], check=True, stderr=subprocess.DEVNULL)
FINAL='music_video.mp4'
if BURN_LYRICS:
    # FontName + fontsdir de libass render dung dau tieng Viet
    style=(f"FontName={FONT_NAME},FontSize=22,PrimaryColour=&H00FFFFFF,"
           f"OutlineColour=&H00000000,Outline=2,Shadow=1,Alignment=2,MarginV=45")
    vf=f"subtitles=lyrics.srt:fontsdir={FONTS_DIR}:force_style='{style}'"
    subprocess.run(['ffmpeg','-y','-i','video_audio.mp4','-vf',vf,'-c:a','copy',FINAL],
                   check=True, stderr=subprocess.DEVNULL)
else:
    os.replace('video_audio.mp4', FINAL)
print('XONG ->', FINAL, '| font:', FONT_NAME)


## 6. Xem & tải video


In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4=open('music_video.mp4','rb').read()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')


In [ ]:
from google.colab import files
files.download('music_video.mp4')


---
### Mẹo
- **Mất dấu tiếng Việt?** Đảm bảo bước 1 in ra `Font phu de: Be Vietnam Pro` (hoặc DejaVu Sans). Cả 2 đều đủ dấu. Tránh font không hỗ trợ (Arial cũ).
- Ảnh đẹp/nét hơn: đổi sang `stabilityai/stable-diffusion-xl-base-1.0` (`num_inference_steps=25, guidance_scale=7`).
- Nhiều cảnh hơn: thêm prompt vào `PROMPTS`.
- TikTok/Reels dọc 9:16: đổi `1280x720`→`720x1280`, `1600:900`→`900:1600`, `crop=1600:900`→`crop=900:1600`.
- 2 video: lần 1 `song_*.wav` + `BURN_LYRICS=True` (có lời/hát); lần 2 `backing_*.wav` + `BURN_LYRICS=False` (không lời).
